# Text to Speech with edge-tts (Bengali and more)

Converts text into natural-sounding speech using Microsoft Edge's online neural voices through the free `edge-tts` library. It lists the available Bengali voices, creates an MP3 you can play in the notebook, and adds a Gradio app, so no ngrok tunnel or token is needed.

## How to run
- Open in Google Colab. Runtime type: **CPU is fine**.
- Run the cells from top to bottom.

_Rewritten from my original Colab notebook: `text to voice.ipynb`._

In [ ]:
!pip install -q edge-tts gradio

## 1. List voices for a language
Colab already runs an event loop, so we can use `await` directly in a cell.

In [ ]:
import edge_tts

voices_manager = await edge_tts.VoicesManager.create()
bn_voices = [v["ShortName"] for v in voices_manager.find(Language="bn")]
print("Bengali voices:", bn_voices)

## 2. Convert text to an MP3

In [ ]:
from IPython.display import Audio, display

async def text_to_speech(text, voice, out_path="speech.mp3", rate="+0%"):
    communicate = edge_tts.Communicate(text, voice, rate=rate)
    await communicate.save(out_path)
    return out_path

text = "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।"
voice = bn_voices[0] if bn_voices else "en-US-AriaNeural"
path = await text_to_speech(text, voice)
display(Audio(path))

## 3. Gradio app
Gradio supports async functions, and `share=True` gives a public link without ngrok.

In [ ]:
import gradio as gr

all_voices = sorted(v["ShortName"] for v in voices_manager.voices)

async def tts_app(text, voice, speed):
    rate = f"{int(speed):+d}%"
    return await text_to_speech(text, voice, "app_output.mp3", rate=rate)

gr.Interface(fn=tts_app,
             inputs=[gr.Textbox(label="Text", lines=4),
                     gr.Dropdown(all_voices, value=voice, label="Voice"),
                     gr.Slider(-50, 50, value=0, step=5, label="Speed (%)")],
             outputs=gr.Audio(type="filepath", label="Speech"),
             title="Text to Speech").launch(share=True)